In [98]:
## read the dataset
dataset = open('input.txt', 'r').read()

In [110]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from collections import Counter

In [ ]:
data = sorted(list(set(dataset)))

In [403]:
vocab_size = 300
initialVocab = len(data)
context_length = 6
batch_size = 4
n_embed = 32
max_iter = 3000
eval_iters = 200
eval_interval = 500
dropout = 0.2

In [102]:
## mapping (char to int) and (int, char)
stoi = {s:i for i,s in enumerate(data)}
itos = {i:s for i,s in enumerate(data)}
encoder = lambda l : [stoi[ch] for ch in l]
decoder = lambda d : [itos[id] for id in d]

In [103]:
## 20% of training data set
n3 = int(0.2 * len(dataset))
text = dataset[:n3]
## creating tokens
tokens = encoder(text)
extravocabs = vocab_size - initialVocab

In [104]:
def get_pair(tokens):

    def create_pairs(tokens):
        counter = Counter()
        for pair in zip(tokens[:], tokens[1:]):
            counter[pair]+=1
        return counter
    counter = create_pairs(tokens)
    max_pair = max(counter, key=counter.get)
    return max_pair

In [105]:
## creating vocabulary using BPE
for j in range(extravocabs):
    pair = get_pair(tokens)
    currentTokenId = j + initialVocab
    i = 0
    new_tokens = []
    while i < len(tokens):
        if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == pair:
            st = itos[tokens[i]]+itos[tokens[i+1]]
            stoi[st] = currentTokenId
            itos[currentTokenId] = st
            new_tokens.append(currentTokenId)
            i+=2
        else:
            new_tokens.append(tokens[i])
            i+=1
    tokens = new_tokens

In [ ]:
## Above is the 300 size vocabulary has been created

In [113]:
## tokenize the whole dataset and split in training and validation
tokenization = torch.tensor(encoder(dataset), dtype=torch.long)
n1 = int(0.9 * len(tokenization))
train_dataset = tokenization[:n1]
val_dataset = tokenization[n1:]

In [ ]:
train_dataset

torch.Size([1003854])

In [117]:
## Next Input and output data split with the batch
torch.manual_seed(1337)
def get_batch(split):
    splitToProcess = train_dataset if split == 'train' else val_dataset
    startingPointers = torch.randint(0, len(splitToProcess)-context_length , (batch_size,))
    x = torch.stack([splitToProcess[i:i+context_length] for i in startingPointers])
    y = torch.stack([splitToProcess[i+1:i+context_length+1] for i in startingPointers])
    return x,y
    

In [148]:
x, y = get_batch('train')

In [404]:
## Creating Transformer block for communication followed by computation
class Block(nn.Module):
    def __init__(self, n_head, n_embed):
        super().__init__()
        self.multiHead = MultiHead(n_head, n_embed//n_head)
        self.feedForward = FeedForward()
        self.layerNorm1 = nn.LayerNorm(n_embed)
        self.layerNorm2 = nn.LayerNorm(n_embed)
    def forward(self, x):
        ## Residual connection
        x = x + self.multiHead(self.layerNorm1(x))
        x = x + self.feedForward(self.layerNorm2(x))
        return x

In [405]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embed, 4* n_embed), nn.ReLU(), nn.Linear(4 * n_embed, n_embed), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

In [406]:
class MultiHead(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [407]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = (q @ k.transpose(-2,-1)) * head_size**-0.5
        tril = torch.tril(torch.ones(T, T))
        wei = wei.masked_fill(tril == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ self.value(x)
        return out

In [408]:
class BiagramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.Embedding = nn.Embedding(vocab_size, n_embed)
        self.ll_head = nn.Linear(n_embed, vocab_size)
        self.positionalEmbedding = nn.Embedding(context_length, n_embed)
        # self.sa_head = Head()
        self.transBlock = nn.Sequential(Block(4, 32),Block(4, 32), Block(4, 32), nn.LayerNorm(n_embed))
    def forward(self, x, out=None):
        B, T = x.shape
        position = self.positionalEmbedding(torch.arange(T))
        token_emb = self.Embedding(x)
        x = token_emb + position
        # x = self.sa_head(x)
        x = self.transBlock(x)
        logits = self.ll_head(x) ## (B,T,vocab_size)
        if out == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            out = out.view(B*T)
            loss = F.cross_entropy(logits,out)
        return logits , loss
    def generate(self, inputToken, max_n_tokens):
        for _ in range(max_n_tokens):
            cropped_inputToken = inputToken[:, -context_length:]
            logits , loss = self(cropped_inputToken)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            pred = torch.multinomial(probs, num_samples = 1)
            inputToken = torch.cat((inputToken, pred), dim=1)
        return inputToken


In [409]:
C = BiagramModel()
out , loss = C.forward(x,y)

In [410]:
print("".join(decoder(C.generate(torch.zeros((1,1), dtype=torch.long), 200)[0].tolist())))


INIUS:
ktaance DERforhat Q, and IRomelld INsar 
 mth:
Chim cen thediLA.

COseyworand y, 'hat pr

gh
Wall:ch inRIOLAolit e
nowhave itiwe uper prsdo qse at atsaKly olitiz the lak himala bld esnmatiour I cijitiztheeemanhads ticez,
TLAyou jRIch earing mleAnd man;
!prat,
Twor,
taneare RIOingwith shilmeverut hat .

COoughd, ereould  theQseRIZr d hof ll ofRIOLAaaentse 3Tbomou ditd, se waou : alK.

COworY, and 's ,ll hat morti'd ce, and Fwith itrsuLAMlvan is itihim d, urDcomor elounas sellCVloin 


In [411]:
## Optimizer
optimizer = torch.optim.AdamW(C.parameters(), lr=1e-3)

In [412]:
@torch.no_grad()
def estimate_loss():
    out = {}
    C.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = C(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    C.train()
    return out

In [ ]:
for iter in range(max_iter):
    ## --- loss estimation 
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f'step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}')
    xb, yb = get_batch('train')

    logits, loss = C(xb,yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step() 

step 0: train loss 2.1625, val loss 2.2120
step 500: train loss 2.1980, val loss 2.2008
step 1000: train loss 2.1766, val loss 2.2405
step 1500: train loss 2.1195, val loss 2.2348
step 2000: train loss 2.1591, val loss 2.1922
step 2500: train loss 2.1371, val loss 2.1666


115782